# 01. Predictive-statistics vertical slice

This notebook visualizes the deterministic mock validation. It is an implementation test, not an astrophysical result.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

path = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "toy_validation" / "toy_validation.json"
report = json.loads(path.read_text())
report["status"], report["comparisons"]

In [ ]:
score_table = pd.DataFrame(
    {name: {k: v for k, v in item.items() if k != "steps"}
     for name, item in report["models"].items()}
).T
score_table

In [ ]:
steps = {}
for name, item in report["models"].items():
    frame = pd.DataFrame(item["steps"])
    steps[name] = frame
delta = steps["peak"]["cumulative_log_score"] - steps["smooth"]["cumulative_log_score"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, len(delta) + 1), delta, marker="o")
ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Chronological held-out event")
ax.set_ylabel(r"$\Delta S_{\rm peak-smooth}$")
ax.set_title("Toy cumulative predictive advantage")
fig.tight_layout()

In [ ]:
pd.DataFrame({
    "event_id": steps["peak"]["event_id"],
    "run": steps["peak"]["run"],
    "delta_increment_peak_smooth": (
        steps["peak"]["log_score_increment"]
        - steps["smooth"]["log_score_increment"]
    ),
    "cumulative_delta_peak_smooth": delta,
})